# Digital Me: consent-gated Discord voice cloning

This notebook receives only samples from Discord users who explicitly run `/voice-consent grant`. Recordings and successful model versions persist in your Google Drive. Recording/upload is continuous; RVC GPU training is asynchronous and serialized.

Before running: choose **Runtime > Change runtime type > T4 GPU**. Keep this tab open while the bot is in use. Colab disconnects are expected; Drive data survives, but an interrupted training job must be restarted.

In [ ]:
from google.colab import drive
from getpass import getpass
from pathlib import Path
import io, os, secrets, subprocess, sys, time, urllib.request, zipfile

drive.mount('/content/drive')
BOT_REPO = 'https://github.com/eikq/DiscordBOT.git'
BOT_BRANCH = 'main'
BOT_ROOT = Path('/content/DiscordBOT')

def move_incomplete_checkout():
    if BOT_ROOT.exists():
        partial_root = BOT_ROOT.with_name(f'{BOT_ROOT.name}.partial-{int(time.time())}')
        BOT_ROOT.rename(partial_root)
        print(f'Moved incomplete checkout to {partial_root}')

def install_repo_snapshot():
    move_incomplete_checkout()
    clone = subprocess.run(
        ['git', '-c', 'http.version=HTTP/1.1', 'clone', '--depth', '1', '--branch', BOT_BRANCH, BOT_REPO, str(BOT_ROOT)],
        text=True, capture_output=True
    )
    if clone.returncode == 0:
        print(clone.stdout.strip() or 'Repository cloned from GitHub.')
        return
    print('git clone failed; falling back to a GitHub ZIP snapshot.')
    print(clone.stderr.strip())
    archive_url = f'https://codeload.github.com/eikq/DiscordBOT/zip/refs/heads/{BOT_BRANCH}'
    with urllib.request.urlopen(archive_url, timeout=120) as response:
        archive_bytes = response.read()
    snapshot_root = Path(f'/content/digital-me-snapshot-{int(time.time())}')
    snapshot_root.mkdir()
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        archive.extractall(snapshot_root)
    extracted_roots = [entry for entry in snapshot_root.iterdir() if entry.is_dir()]
    if len(extracted_roots) != 1:
        raise RuntimeError('GitHub ZIP did not contain one repository root.')
    extracted_roots[0].rename(BOT_ROOT)
    snapshot_root.rmdir()
    print('Repository downloaded from GitHub ZIP snapshot.')

if (BOT_ROOT / '.git').is_dir():
    pull = subprocess.run(['git', '-c', 'http.version=HTTP/1.1', 'pull', '--ff-only'], cwd=BOT_ROOT, text=True, capture_output=True)
    if pull.returncode != 0:
        print('git pull failed; refreshing from a clean snapshot.')
        print(pull.stderr.strip())
        install_repo_snapshot()
else:
    install_repo_snapshot()

token = getpass('Shared API token (leave empty to generate a strong one): ').strip()
if not token:
    token = secrets.token_urlsafe(36)
if len(token) < 24:
    raise ValueError('Use a token containing at least 24 characters.')
os.environ['COLAB_API_TOKEN'] = token
os.environ['VOICE_DRIVE_ROOT'] = '/content/drive/MyDrive/DigitalMeVoice'
os.environ['RVC_ROOT'] = '/content/Retrieval-based-Voice-Conversion-WebUI'
print('Drive mounted and service secret configured.')

In [ ]:
# The first run downloads the pinned RVC code, CUDA packages, and model assets.
subprocess.run([sys.executable, str(BOT_ROOT / 'colab' / 'bootstrap_colab.py')], check=True)

In [ ]:
import re, requests, threading, time

for process_name in ('VOICE_SERVER', 'VOICE_TUNNEL'):
    previous = globals().get(process_name)
    if previous and previous.poll() is None:
        previous.terminate()

service_log = open('/content/digital_me_voice_service.log', 'a', buffering=1)
VOICE_SERVER = subprocess.Popen(
    [sys.executable, str(BOT_ROOT / 'colab' / 'voice_service.py')],
    env=os.environ.copy(), stdout=service_log, stderr=subprocess.STDOUT, text=True
)
for _ in range(90):
    try:
        if requests.get('http://127.0.0.1:8766/health', timeout=1).ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError('Voice service did not start. Inspect /content/digital_me_voice_service.log')

VOICE_TUNNEL = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8766', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
public_url = None
captured = []
deadline = time.time() + 60
while time.time() < deadline:
    line = VOICE_TUNNEL.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    captured.append(line.rstrip())
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    raise RuntimeError('Cloudflare URL was not created:\n' + '\n'.join(captured[-20:]))

def drain_tunnel():
    for _ in VOICE_TUNNEL.stdout:
        pass
threading.Thread(target=drain_tunnel, daemon=True).start()

print('\nCOLAB_VOICE_URL=' + public_url)
print('COLAB_API_TOKEN=' + os.environ['COLAB_API_TOKEN'])
print('\nCopy both values into the bot .env, set RECORD_RAW_AUDIO=true, then restart the bot.')
print('Service log: /content/digital_me_voice_service.log')

## Discord flow

1. Each participant who wants their own model runs `/voice-consent grant`.
2. The bot owner joins with `/join`; eligible utterances upload to Drive immediately.
3. Automatic first training begins at 600 seconds by default. For an experimental early model, the user can run `/voice-train start` after 120 seconds.
4. Check with `/voice-train status`, choose a ready voice with `/voice user:@name`, and test with `/speak`.
5. `/voice-consent revoke` stops future capture. `/voice-consent delete` removes local files and asks this service to delete that user's Drive samples and models.